## MSETAR Experiments

This section contains the corresponding simulation procedures and forecasting experiments.
Users can either generate new synthetic datasets by selecting the simulation parameters or load existing datasets stored in the Data/MSETAR folder. The provided datasets correspond to the experiments conducted in the thesis and facilitate the replication of the reported results.

### Import Required Libraries

Run the Required Libraries before executing any simulation or forecasting experiment.

In [1]:
import os
os.chdir(r"xxx")
from model.Base import Base
from model.MSVR import MSVR
from model.utility import (
    create_dataset,
    create_dataset_antes,
    rmse,
    CustomMSVR,
    create_dataset_rez,
    rezago_sig
)

from scikeras.wrappers import KerasRegressor

from scipy.linalg import orth
from scipy.stats import multivariate_normal

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit,
    train_test_split
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank

import csv
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

### Option 1: Generate Synthetic Data and Run the forecasting experiment
This option generates synthetic datasets from a two-regime Multivariate Self-Exciting Threshold Autoregressive (MSETAR) process. Users can customize the simulation settings directly in the R script (MSETAR_R) before running the experiment. The generated datasets are stored in Excel files and can subsequently be used for model fitting and forecast evaluation.

Before running the simulation, the user can define the following parameters:

| Parameter | Description |
|-----------|-------------|
| long | Length of the generated time series. |
| k | Dimension of the multivariate process. |
| p | Number of autoregressive lags. |
| h.ahead | Forecast horizon (steps ahead). |
| iterations | Number of Monte Carlo replications. |
| phi1, phi2 | Regime-specific autoregressive coefficient matrices. |
| sigma1, sigma2 | Regime-specific innovation covariance matrices. |
| delay | Delay applied to the threshold variable. |
| Trim | Trimming bounds for regime switching. |
| umbral | Threshold value used to determine the active regime. |



Example:

long <- c(200, 500, 1000, 5000)  # Length of the series 

h.ahead <- 1                      # Forecast horizon (steps ahead)

k <- 2                            # Number of variables

iterations <- 300                 # Number of simulation replications

p <- 1                            # Number of lags

phi1 <- matrix(c(0.5, 0.7,
                 0.3, 0.2), k, k)

phi2 <- matrix(c(-0.4, -0.6,
                 -0.5,  0.5), k, k)

sigma1 <- matrix(c(1, 0,
                   0, 1), 2, 2)

sigma2 <- matrix(c(1, 0,
                   0, 1), 2, 2)

c1 <- c(0, 0)

c2 <- c(0, 0)

delay <- c(1, 1)  # Delay applied to the threshold variable used for regime switching

Trim <- c(0.2, 0.8)  # Trimming bounds used to avoid excessively frequent transitions between regimes

umbral <- 0  # Threshold value that determines the active regime

After the datasets have been generated in R, they can be loaded into the Python framework to fit forecasting models and evaluate predictive performance. 

### MSETAR_CONFIG

The `MSETAR_CONFIG` script contains all the functionality required to run the simulation study. Specifically, it includes:

- Generation of synthetic datasets from the MSETAR process under different simulation settings.
- Configuration of sample sizes, model parameters, and forecasting horizons.
- Estimation and evaluation of the proposed MSETAR forecasting framework.
- Implementation of the classical forecasting model used as a benchmark for comparison.
- Computation of performance metrics across all simulation replications.

Users can modify the simulation parameters directly in this script before running the experiments.


## Run the forecasting model

In [ ]:
t = [50,200,500,1000,5000]
k = 2     # Dimension of the vector Y 
p = 1     # Number of lags 
h = 1     # Forecast horizon 

import warnings
warnings.filterwarnings("ignore")

rez=1
hiperparametros_svr = {size: [] for size in t}
vectores_soporte = {size: [] for size in t}
train_RMSE_svr = {size: [] for size in t}
test_RMSE_svr = {size: [] for size in t}
tiempo_msvr = {size: [] for size in t}
resultados = []


data_folder = "xxx"
def load_data(size, no):
    file_path = f"{data_folder}/dataset_size_{size}.xlsx"
    sheet_name = f"Iter_{no}"
    data = pd.read_excel(file_path, sheet_name=sheet_name)
    return data



for size in t:
      a=size
      for no in range(100):
            print(f"-------------------------Size {a}-------------------------")
            print(f"-------------------------Iteration {no}--------------------------")

            # Read series
            series = load_data(size, no)
        # --------------------------------
        # Forecasting experiment
        # --------------------------------
            
             # Fit the MSVR model
            start_time = time.time()
            fechas = pd.DataFrame(list(range(len(series))))
            total = pd.concat([fechas,series], axis=1).values
            dim=len(total)
            #print(dim)
            #print(total)
            #Dataset construction
            data=Base(total)
            data= data.base
            #Create the supervised learning dataset
            dataset = create_dataset_rez(data,dim,h,k,p)
            X, Y = dataset[:, :(0 - h*k)], dataset[:, (0-h*k):]
            #Train-test split
            X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, shuffle=False)
            #Feature standardization
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()
            scaler_X.fit(X_train)
            scaler_y.fit(y_train)
            X_train_nor = scaler_X.transform(X_train)
            X_test_nor = scaler_X.transform(X_test)
            y_train_nor = scaler_y.transform(y_train)
            y_test_nor = scaler_y.transform(y_test)
            pipe = Pipeline([
                ('MSVR', CustomMSVR(kernel='rbf', degree=3, gamma=0, coef0=0.0, tol=0.001, C=1.0, epsilon=0.1))
            ])
            hyperparameters = {
                #'MSVR__kernel': ['poly'],
                'MSVR__kernel': ['poly','rbf','linear'],
                'MSVR__degree': [2,5],
                #'MSVR__degree': [1],
                'MSVR__gamma': [0.5,1],
                'MSVR__coef0': [0.1,0.5,1],
                'MSVR__C': [5,9,11,13],
                'MSVR__epsilon':[1,2],
            }

            #Con tscv 
            tscv=TimeSeriesSplit(n_splits=5)
            bm_tscv = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=tscv, verbose=0, error_score='raise')
            best_model_tscv = bm_tscv.fit(X_train_nor, y_train_nor)
            best_params_tscv = bm_tscv.best_params_
            msvr_tscv = MSVR(kernel=bm_tscv.best_params_.get("MSVR__kernel"), gamma=bm_tscv.best_params_.get("MSVR__gamma"),
                            epsilon=bm_tscv.best_params_.get("MSVR__epsilon"), C=bm_tscv.best_params_.get("MSVR__C"),
                            degree=bm_tscv.best_params_.get("MSVR__degree"), coef0=bm_tscv.best_params_.get("MSVR__coef0"), tol=0.01)
            msvr_tscv.fit(X_train_nor, y_train_nor)
            trainPred_svr_nor_tscv = msvr_tscv.predict(X_train_nor)
            testPred_svr_nor_tscv = msvr_tscv.predict(X_test_nor)
            trainPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor_tscv))
            testPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor_tscv))
            train_rmse_svr_tscv = rmse(y_train, trainPred_svr_tscv)
            test_rmse_svr_tscv = rmse(y_test, testPred_svr_tscv)
            hiperparametros_svr_tscv[size].append(best_params_tscv)
            vectores_soporte_tscv[size].append(msvr_tscv.NSV)
            train_RMSE_svr_tscv[size].append(train_rmse_svr_tscv)
            test_RMSE_svr_tscv[size].append(test_rmse_svr_tscv)
            end_time = time.time()
            execution_time_tscv = end_time - start_time
            tiempo_msvr_tscv[size].append(end_time - start_time) 

            #Sin tscv 
            bm = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=5, verbose=0, error_score='raise', random_state=42)
            best_model = bm.fit(X_train_nor, y_train_nor)
            best_params = bm.best_params_
            msvr = MSVR(kernel=bm.best_params_.get("MSVR__kernel"), gamma=bm.best_params_.get("MSVR__gamma"),
                            epsilon=bm.best_params_.get("MSVR__epsilon"), C=bm.best_params_.get("MSVR__C"),
                            degree=bm.best_params_.get("MSVR__degree"), coef0=bm.best_params_.get("MSVR__coef0"), tol=0.01)
            msvr.fit(X_train_nor, y_train_nor)
            trainPred_svr_nor = msvr.predict(X_train_nor)
            testPred_svr_nor = msvr.predict(X_test_nor)
            trainPred_svr  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor))
            testPred_svr  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor))
            train_rmse_svr = rmse(y_train, trainPred_svr)
            #print(train_rmse_svr)
            test_rmse_svr = rmse(y_test, testPred_svr)
            hiperparametros_svr[size].append(best_params)
            vectores_soporte[size].append(msvr.NSV)
            train_RMSE_svr[size].append(train_rmse_svr)
            test_RMSE_svr[size].append(test_rmse_svr)
            end_time = time.time()
            execution_time = end_time - start_time
            tiempo_msvr[size].append(end_time - start_time)
            print("VAR DIF Train RMSE:", train_rmse_var_dif, "Test RMS ", test_rmse_var_dif)
            print("VEC Train RMSE:", train_rmse_vec, "Test RMS ", test_rmse_vec)
            print("MSVR Train RMSE:", train_rmse_svr, "Test RMS ", test_rmse_svr)
            print("MSVR_tscv Train RMSE:", train_rmse_svr_tscv, "MSVR_tscv ", test_rmse_svr_tscv)

filename = f'xxxxx.csv'
with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Tamaño', 'Modelo', 'Hiperparametros', 'Vectores_soporte', 'Train RMSE', 'Test RMSE', 'Tiempo'])
        for size in t:
            writer.writerow([f'Tamaño: {size}', '', '', '', '', '', ''])
            for i in range(100):
                writer.writerow(['SVR', hiperparametros_svr[size][i], vectores_soporte[size][i], train_RMSE_svr[size][i], test_RMSE_svr[size][i], tiempo_msvr[size][i]])
            
            
                
            print(f'Results saved in {filename}')
        
